In [1]:
import io
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split

from collections import Counter

In [2]:
# ============================================
# Configuration
# ============================================

SAMPLE_RATE = 16000

FRAME_LENGTH = 0.025
FRAME_HOP = 0.020

N_FFT = 400
HOP_LENGTH = 320
N_MELS = 80

CHUNK_DURATION = 1.5
CHUNK_SAMPLES = int(
    SAMPLE_RATE * CHUNK_DURATION
)

SHARD_SIZE = 2000

PARQUET_FILES = sorted(
    glob.glob(
        "../data/raw/librispeech-segment/"
        "train.clean.100-*.parquet"
    )
)

print(f"Using {len(PARQUET_FILES)} parquet files")

Using 15 parquet files


In [3]:
# ============================================
# Phoneme vocabulary
# ============================================

phone_counts = Counter()

for path in PARQUET_FILES:

    table = pq.read_table(
        path,
        columns=["phones"]
    )

    for phone_list in table["phones"].to_pylist():
        phone_counts.update(phone_list)

phones = sorted(phone_counts)

phone_to_id = {
    phone: i
    for i, phone in enumerate(phones)
}

id_to_phone = {
    i: phone
    for phone, i in phone_to_id.items()
}

NUM_CLASSES = len(phones)

print("Number of classes:", NUM_CLASSES)
print(phones)

Number of classes: 41
['AA', 'AE', 'AH', 'AO', 'AW', 'AY', 'B', 'CH', 'D', 'DH', 'EH', 'ER', 'EY', 'F', 'G', 'HH', 'IH', 'IY', 'JH', 'K', 'L', 'M', 'N', 'NG', 'OW', 'OY', 'P', 'R', 'S', 'SH', 'T', 'TH', 'UH', 'UW', 'V', 'W', 'Y', 'Z', 'ZH', '[SIL]', '[UNK]']


In [4]:
# ============================================
# Speaker-level train / validation / test split
# ============================================

speaker_ids = []

for path in PARQUET_FILES:

    table = pq.read_table(
        path,
        columns=["speaker_id"]
    )

    speaker_ids.extend(
        table["speaker_id"].to_pylist()
    )

speakers = np.array(
    sorted(set(speaker_ids))
)

# 20% testing
trainval_speakers, test_speakers = train_test_split(
    speakers,
    test_size=0.20,
    random_state=42
)

# 10% validation
train_speakers, val_speakers = train_test_split(
    trainval_speakers,
    test_size=0.10,
    random_state=42
)

train_speakers = set(train_speakers)
val_speakers = set(val_speakers)
test_speakers = set(test_speakers)

print("Total speakers:", len(speakers))
print("Training speakers:", len(train_speakers))
print("Validation speakers:", len(val_speakers))
print("Test speakers:", len(test_speakers))

Total speakers: 251
Training speakers: 180
Validation speakers: 20
Test speakers: 51


In [5]:
# ============================================
# Build chunk indexes
# ============================================

train_chunk_index = []
val_chunk_index = []
test_chunk_index = []

for file_id, path in enumerate(PARQUET_FILES):

    table = pq.read_table(
        path,
        columns=[
            "speaker_id",
            "duration"
        ]
    )

    speakers_in_file = (
        table["speaker_id"].to_pylist()
    )

    durations_in_file = (
        table["duration"].to_pylist()
    )

    for row_id, (
        speaker_id,
        duration
    ) in enumerate(
        zip(
            speakers_in_file,
            durations_in_file
        )
    ):

        num_chunks = int(
            np.ceil(
                duration / CHUNK_DURATION
            )
        )

        for chunk_id in range(num_chunks):

            item = {
                "file_id": file_id,
                "row_id": row_id,
                "chunk_id": chunk_id,
            }

            if speaker_id in train_speakers:

                train_chunk_index.append(item)

            elif speaker_id in val_speakers:

                val_chunk_index.append(item)

            else:

                test_chunk_index.append(item)

print(
    "Training chunks:",
    len(train_chunk_index)
)

print(
    "Validation chunks:",
    len(val_chunk_index)
)

print(
    "Test chunks:",
    len(test_chunk_index)
)

Training chunks: 184840
Validation chunks: 19754
Test chunks: 51098


In [6]:
# ============================================
# Output directories
# ============================================

PROCESSED_DIR = (
    "../data/processed/phoneme_chunks_1500ms"
)

TRAIN_DIR = (
    f"{PROCESSED_DIR}/train"
)

VAL_DIR = (
    f"{PROCESSED_DIR}/validation"
)

TEST_DIR = (
    f"{PROCESSED_DIR}/test"
)

import os

os.makedirs(
    TRAIN_DIR,
    exist_ok=True
)

os.makedirs(
    VAL_DIR,
    exist_ok=True
)

os.makedirs(
    TEST_DIR,
    exist_ok=True
)

print(
    "Processed data directories ready."
)

Processed data directories ready.


In [7]:
# ============================================
# Audio decoding
# ============================================

def process_audio(audio_entry):

    audio, sample_rate = sf.read(
        io.BytesIO(
            audio_entry["bytes"]
        ),
        dtype="float32"
    )

    if audio.ndim > 1:

        audio = np.mean(
            audio,
            axis=1
        )

    if sample_rate != SAMPLE_RATE:

        audio = librosa.resample(
            audio,
            orig_sr=sample_rate,
            target_sr=SAMPLE_RATE
        )

    return audio

In [8]:
# ============================================
# Create one 1-second feature chunk
# ============================================

def create_chunk(
    audio,
    phones,
    phone_starts,
    phone_ends,
    chunk_id,
):

    chunk_start_sample = (
        chunk_id * CHUNK_SAMPLES
    )

    chunk_end_sample = (
        chunk_start_sample
        + CHUNK_SAMPLES
    )

    chunk = audio[
        chunk_start_sample:chunk_end_sample
    ]

    actual_samples = len(chunk)

    # Pad final chunk if necessary
    if actual_samples < CHUNK_SAMPLES:

        chunk = np.pad(
            chunk,
            (
                0,
                CHUNK_SAMPLES - actual_samples
            )
        )

    # Log-Mel spectrogram
    log_mel = (
        librosa.feature.melspectrogram(
            y=chunk,
            sr=SAMPLE_RATE,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
            power=2.0,
            center=False,
        )
    )

    log_mel = librosa.power_to_db(
        log_mel,
        ref=np.max
    )

    # Time boundaries
    chunk_start_time = (
        chunk_start_sample
        / SAMPLE_RATE
    )

    actual_audio_end = (
        chunk_start_time
        + actual_samples
        / SAMPLE_RATE
    )

    labels = []

    # Label every feature frame
    for frame_index in range(
        log_mel.shape[1]
    ):

        frame_start = (
            chunk_start_time
            + frame_index * FRAME_HOP
        )

        frame_end = (
            frame_start
            + FRAME_LENGTH
        )

        # Ignore padded portion
        if frame_start >= actual_audio_end:

            labels.append(-100)

            continue

        best_phone = "[UNK]"
        best_overlap = 0.0

        # Find phone with greatest overlap
        for (
            phone,
            phone_start,
            phone_end
        ) in zip(
            phones,
            phone_starts,
            phone_ends
        ):

            overlap_start = max(
                frame_start,
                phone_start
            )

            overlap_end = min(
                frame_end,
                phone_end
            )

            overlap = max(
                0.0,
                overlap_end - overlap_start
            )

            if overlap > best_overlap:

                best_overlap = overlap
                best_phone = phone

        labels.append(
            phone_to_id[best_phone]
        )

    return (
        np.asarray(
            log_mel,
            dtype=np.float32
        ),
        np.asarray(
            labels,
            dtype=np.int64
        )
    )

In [9]:
# ============================================
# Sharded preprocessing
# ============================================

def preprocess_split(
    chunk_index,
    output_dir,
    split_name,
):

    output_dir = str(output_dir)

    chunks_by_file = {}

    for item in chunk_index:

        file_id = item["file_id"]

        if file_id not in chunks_by_file:

            chunks_by_file[file_id] = []

        chunks_by_file[file_id].append(
            item
        )

    total_chunks = len(
        chunk_index
    )

    processed_chunks = 0

    shard_X = []
    shard_y = []

    shard_id = 0

    print(
        f"Starting {split_name} preprocessing"
    )

    print(
        f"Total chunks: {total_chunks}"
    )

    for file_id in sorted(
        chunks_by_file
    ):

        path = PARQUET_FILES[
            file_id
        ]

        print(
            f"\nProcessing "
            f"{os.path.basename(path)}"
        )

        table = pq.read_table(
            path,
            columns=[
                "audio",
                "phones",
                "phone_starts",
                "phone_ends",
            ]
        )

        chunks_by_row = {}

        for item in chunks_by_file[
            file_id
        ]:

            row_id = item["row_id"]

            if row_id not in chunks_by_row:

                chunks_by_row[row_id] = []

            chunks_by_row[row_id].append(
                item["chunk_id"]
            )

        for row_id, chunk_ids in (
            chunks_by_row.items()
        ):

            audio_entry = (
                table["audio"][
                    row_id
                ].as_py()
            )

            audio = process_audio(
                audio_entry
            )

            phones = (
                table["phones"][
                    row_id
                ].as_py()
            )

            phone_starts = (
                table["phone_starts"][
                    row_id
                ].as_py()
            )

            phone_ends = (
                table["phone_ends"][
                    row_id
                ].as_py()
            )

            for chunk_id in sorted(
                chunk_ids
            ):

                X, y = create_chunk(
                    audio=audio,
                    phones=phones,
                    phone_starts=phone_starts,
                    phone_ends=phone_ends,
                    chunk_id=chunk_id,
                )

                shard_X.append(X)
                shard_y.append(y)

                processed_chunks += 1

                # Save shard
                if len(shard_X) >= SHARD_SIZE:

                    shard_path = os.path.join(
                        output_dir,
                        f"shard_{shard_id:05d}.npz"
                    )

                    np.savez_compressed(
                        shard_path,
                        X=np.stack(shard_X),
                        y=np.stack(shard_y),
                    )

                    print(
                        f"Saved "
                        f"{os.path.basename(shard_path)} "
                        f"({len(shard_X)} chunks)"
                    )

                    shard_X = []
                    shard_y = []

                    shard_id += 1

        del table

        print(
            f"Processed "
            f"{processed_chunks}/"
            f"{total_chunks}"
        )

    # Save final partial shard
    if shard_X:

        shard_path = os.path.join(
            output_dir,
            f"shard_{shard_id:05d}.npz"
        )

        np.savez_compressed(
            shard_path,
            X=np.stack(shard_X),
            y=np.stack(shard_y),
        )

        print(
            f"Saved "
            f"{os.path.basename(shard_path)} "
            f"({len(shard_X)} chunks)"
        )

    print(
        f"\nFinished {split_name}"
    )

    print(
        f"Total chunks saved: "
        f"{processed_chunks}"
    )

    return processed_chunks

In [10]:
# ============================================
# Process training data
# ============================================

train_count = preprocess_split(
    chunk_index=train_chunk_index,
    output_dir=TRAIN_DIR,
    split_name="TRAIN",
)

Starting TRAIN preprocessing
Total chunks: 184840

Processing train.clean.100-00000-of-00015.parquet


c:\Users\louiz\Documents\Projects\phoneme-lipsync\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved shard_00000.npz (2000 chunks)
Saved shard_00001.npz (2000 chunks)
Saved shard_00002.npz (2000 chunks)
Saved shard_00003.npz (2000 chunks)
Saved shard_00004.npz (2000 chunks)
Saved shard_00005.npz (2000 chunks)
Processed 12657/184840

Processing train.clean.100-00001-of-00015.parquet
Saved shard_00006.npz (2000 chunks)
Saved shard_00007.npz (2000 chunks)
Saved shard_00008.npz (2000 chunks)
Saved shard_00009.npz (2000 chunks)
Saved shard_00010.npz (2000 chunks)
Saved shard_00011.npz (2000 chunks)
Saved shard_00012.npz (2000 chunks)
Processed 26116/184840

Processing train.clean.100-00002-of-00015.parquet
Saved shard_00013.npz (2000 chunks)
Saved shard_00014.npz (2000 chunks)
Saved shard_00015.npz (2000 chunks)
Saved shard_00016.npz (2000 chunks)
Saved shard_00017.npz (2000 chunks)
Saved shard_00018.npz (2000 chunks)
Saved shard_00019.npz (2000 chunks)
Saved shard_00020.npz (2000 chunks)
Processed 42866/184840

Processing train.clean.100-00003-of-00015.parquet
Saved shard_00021.npz 

In [11]:
# ============================================
# Process validation data
# ============================================

val_count = preprocess_split(
    chunk_index=val_chunk_index,
    output_dir=VAL_DIR,
    split_name="VALIDATION",
)

Starting VALIDATION preprocessing
Total chunks: 19754

Processing train.clean.100-00000-of-00015.parquet
Processed 1779/19754

Processing train.clean.100-00001-of-00015.parquet
Saved shard_00000.npz (2000 chunks)
Processed 3769/19754

Processing train.clean.100-00003-of-00015.parquet
Saved shard_00001.npz (2000 chunks)
Processed 4837/19754

Processing train.clean.100-00004-of-00015.parquet
Saved shard_00002.npz (2000 chunks)
Processed 7936/19754

Processing train.clean.100-00005-of-00015.parquet
Saved shard_00003.npz (2000 chunks)
Processed 9769/19754

Processing train.clean.100-00006-of-00015.parquet
Saved shard_00004.npz (2000 chunks)
Saved shard_00005.npz (2000 chunks)
Processed 12560/19754

Processing train.clean.100-00008-of-00015.parquet
Saved shard_00006.npz (2000 chunks)
Processed 14686/19754

Processing train.clean.100-00009-of-00015.parquet
Processed 15751/19754

Processing train.clean.100-00010-of-00015.parquet
Saved shard_00007.npz (2000 chunks)
Processed 17826/19754

Proce

In [12]:
# ============================================
# Process test data
# ============================================

test_count = preprocess_split(
    chunk_index=test_chunk_index,
    output_dir=TEST_DIR,
    split_name="TEST",
)

Starting TEST preprocessing
Total chunks: 51098

Processing train.clean.100-00000-of-00015.parquet
Saved shard_00000.npz (2000 chunks)
Processed 3494/51098

Processing train.clean.100-00001-of-00015.parquet
Saved shard_00001.npz (2000 chunks)
Saved shard_00002.npz (2000 chunks)
Processed 6255/51098

Processing train.clean.100-00002-of-00015.parquet
Processed 7429/51098

Processing train.clean.100-00003-of-00015.parquet
Saved shard_00003.npz (2000 chunks)
Saved shard_00004.npz (2000 chunks)
Saved shard_00005.npz (2000 chunks)
Processed 13224/51098

Processing train.clean.100-00004-of-00015.parquet
Saved shard_00006.npz (2000 chunks)
Saved shard_00007.npz (2000 chunks)
Processed 17512/51098

Processing train.clean.100-00005-of-00015.parquet
Saved shard_00008.npz (2000 chunks)
Saved shard_00009.npz (2000 chunks)
Processed 20275/51098

Processing train.clean.100-00006-of-00015.parquet
Saved shard_00010.npz (2000 chunks)
Saved shard_00011.npz (2000 chunks)
Processed 25046/51098

Processing 

In [13]:
# ============================================
# Verify processed shards
# ============================================

TIME_FRAMES = (
    (CHUNK_SAMPLES - N_FFT) // HOP_LENGTH
) + 1

print(f"Expected time frames per chunk: {TIME_FRAMES}")


def verify_split(
    directory,
    expected_chunks,
    name,
):

    shard_files = sorted(
        glob.glob(
            os.path.join(
                directory,
                "shard_*.npz"
            )
        )
    )

    total_chunks = 0

    for path in shard_files:

        with np.load(path) as shard:

            X = shard["X"]
            y = shard["y"]

            assert X.shape[0] == y.shape[0]

            assert X.shape[1:] == (
                N_MELS,
                TIME_FRAMES
            )

            assert y.shape[1] == TIME_FRAMES

            total_chunks += X.shape[0]

    print(f"\n{name}")
    print("-" * 30)
    print(
        "Shards:",
        len(shard_files)
    )
    print(
        "Chunks:",
        total_chunks
    )
    print(
        "Expected:",
        expected_chunks
    )
    print(
        "Match:",
        total_chunks == expected_chunks
    )


verify_split(
    TRAIN_DIR,
    len(train_chunk_index),
    "TRAIN"
)

verify_split(
    VAL_DIR,
    len(val_chunk_index),
    "VALIDATION"
)

verify_split(
    TEST_DIR,
    len(test_chunk_index),
    "TEST"
)

Expected time frames per chunk: 74

TRAIN
------------------------------
Shards: 93
Chunks: 184840
Expected: 184840
Match: True

VALIDATION
------------------------------
Shards: 10
Chunks: 19754
Expected: 19754
Match: True

TEST
------------------------------
Shards: 26
Chunks: 51098
Expected: 51098
Match: True


In [14]:
# ============================================
# Final preprocessing summary
# ============================================

print("PREPROCESSING COMPLETE")
print("=" * 40)

print(
    "Training chunks:",
    train_count
)

print(
    "Validation chunks:",
    val_count
)

print(
    "Test chunks:",
    test_count
)

print(
    "\nChunk shape:",
    f"({N_MELS}, {TIME_FRAMES})"
)

print(
    "Chunk duration:",
    f"{CHUNK_DURATION:.1f} seconds"
)

print(
    "Shard size:",
    SHARD_SIZE
)

print(
    "Phoneme classes:",
    NUM_CLASSES
)

print(
    "\nOutput:",
    os.path.abspath(PROCESSED_DIR)
)

PREPROCESSING COMPLETE
Training chunks: 184840
Validation chunks: 19754
Test chunks: 51098

Chunk shape: (80, 74)
Chunk duration: 1.5 seconds
Shard size: 2000
Phoneme classes: 41

Output: c:\Users\louiz\Documents\Projects\phoneme-lipsync\data\processed\phoneme_chunks_1500ms
